# gold_payment_lifecycle

**Source:** `silver.payment_events`  
**Grain:** one row per `payment_id` — full event lifecycle summary  
**Merge key:** `payment_id`  
**Lineage:** Silver → Gold (never Bronze directly)

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "ubereats_dev")

In [ ]:
catalog    = dbutils.widgets.get("catalog")
gold_table = f"{catalog}.gold.payment_lifecycle"
silver_src = f"{catalog}.silver.payment_events"

print(f"[gold] {gold_table}  ←  {silver_src}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_table} (
        payment_id             STRING NOT NULL,
        event_count            LONG,
        first_event_name       STRING,
        last_event_name        STRING,
        first_event_at         TIMESTAMP,
        last_event_at          TIMESTAMP,
        lifecycle_duration_sec DOUBLE,
        _computed_at           TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (payment_id)
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[gold] table {gold_table} ready")

In [ ]:
from pyspark.sql.functions import (
    col, coalesce, get_json_object,
    count, min, max, first, last,
    unix_timestamp, current_timestamp,
)

# fact: silver.payment_events
lifecycle_df = (
    spark.table(silver_src)
    .withColumn(
        "event_name",
        coalesce(get_json_object(col("event"), "$.event_name"), col("event")),
    )
    .groupBy("payment_id")
    .agg(
        count("event_id").alias("event_count"),
        first("event_name", ignorenulls=True).alias("first_event_name"),
        last("event_name",  ignorenulls=True).alias("last_event_name"),
        min("dt_current_timestamp").alias("first_event_at"),
        max("dt_current_timestamp").alias("last_event_at"),
    )
    .withColumn(
        "lifecycle_duration_sec",
        (unix_timestamp("last_event_at") - unix_timestamp("first_event_at")).cast("double"),
    )
    .withColumn("_computed_at", current_timestamp())
)

print(f"[gold] {lifecycle_df.count()} payment lifecycles")

In [ ]:
lifecycle_df.createOrReplaceTempView("gold_payment_lifecycle_batch")

spark.sql(f"""
    MERGE INTO {gold_table} AS t
    USING gold_payment_lifecycle_batch AS s
    ON t.payment_id = s.payment_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"[gold] {gold_table} updated")